In [22]:
import pandas as pd
import csv
import matplotlib.pyplot as plt

In [23]:
def parse_params(params_str):
    params = {}
    for item in params_str.split():
        if '=' in item:
            key, value = item.split('=', 1)
            params[key] = value
    return params

# aggregates results over all parameters except those in params_to_compare
def results_csv_to_df(csv_file, params_to_aggregate=None):
    results = []

    with open(csv_file, newline='') as f:
        reader = csv.DictReader(f, fieldnames=[
            'accuracy', 'correctly_filled_cells', 'nfe', 'time', 'speed', 'not_fully_unmasked',
            'checkpoint', 'strategy', 'params'
        ])
        reader.__next__()  # Skip header row
        for row in reader:
            params = parse_params(row['params'])
            row_dict = {
                'accuracy': float(row['accuracy']),
                'correctly_filled_cells': float(row['correctly_filled_cells']),
                'checkpoint': row['checkpoint'],
                'strategy': row['strategy'],
                'nfe': float(row['nfe']),
                'time': float(row['time']),
                'speed': float(row['speed']),
            }
            row_dict.update(params.items())
            results.append(row_dict)
    
    results_df = pd.DataFrame(results)
    print(f"Total rows before aggregation: {len(results_df)}")
    
    # Aggregate the accuracy, correctly_filled_cells, nfe, time, and speed over the rows where all parameters except those in params_to_aggregate are the same
    if params_to_aggregate is not None:
        groupby_cols = [col for col in results_df.columns if col not in params_to_aggregate + ['accuracy', 'correctly_filled_cells', 'nfe', 'time', 'speed']]
    else:
        groupby_cols = [col for col in results_df.columns if col not in ['accuracy', 'correctly_filled_cells', 'nfe', 'time', 'speed']]
    
    aggregated_df = results_df.groupby(groupby_cols, dropna=False).agg(
        accuracy_mean=pd.NamedAgg(column='accuracy', aggfunc='mean'),
        accuracy_std=pd.NamedAgg(column='accuracy', aggfunc='std'),
        correctly_filled_cells_mean=pd.NamedAgg(column='correctly_filled_cells', aggfunc='mean'),
        correctly_filled_cells_std=pd.NamedAgg(column='correctly_filled_cells', aggfunc='std'),
        nfe=pd.NamedAgg(column='nfe', aggfunc='mean'),
        time=pd.NamedAgg(column='time', aggfunc='mean'),
        speed=pd.NamedAgg(column='speed', aggfunc='mean'),
    ).reset_index()

    print(f"Rows after aggregation: {len(aggregated_df)}")
    return aggregated_df

In [24]:
params_to_aggregate = ['seed']
df = results_csv_to_df('results_presentation_adaptive.csv', params_to_aggregate=params_to_aggregate)
df

Total rows before aggregation: 18
Rows after aggregation: 6


,checkpoint,strategy,score_mask_position,select_position,unmask_token,k,gumbel_noise_coefficient,dataset,num_samples,num_denoising_steps,...,change_token,self_correction,select_position_unmask,accuracy_mean,accuracy_std,correctly_filled_cells_mean,correctly_filled_cells_std,nfe,time,speed
0,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,NaN,top_k_gumbel,NaN,1,0,easy,6400,80,...,change_max,none,NaN,0.970767,0.002103,0.987500,0.000700,1.083333,252.666667,25.353333
1,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,NaN,top_k_gumbel,NaN,1,0,hard,6400,80,...,change_max,none,NaN,0.679533,0.003350,0.854267,0.001914,1.250000,315.000000,20.316667
2,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,MDM_max,NaN,MDM_max,1,0,easy,6400,80,...,change_max,NaN,top_k_gumbel,0.967033,0.002335,0.987133,0.000764,1.213333,300.666667,21.296667
3,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,MDM_max,NaN,MDM_max,1,0,hard,6400,80,...,change_max,NaN,top_k_gumbel,0.665200,0.004419,0.852767,0.001877,1.250000,329.666667,19.510000
4,checkpoints/mdlm/300_epochs,mdlm_adaptive_score_select_update,MDM_max,top_k_gumbel,MDM_max,1,0,easy,6400,80,...,NaN,NaN,NaN,0.983967,0.001301,0.993767,0.000503,0.920000,97.666667,65.553333
5,checkpoints/mdlm/300_epochs,mdlm_adaptive_score_select_update,MDM_max,top_k_gumbel,MDM_max,1,0,hard,6400,80,...,NaN,NaN,NaN,0.652300,0.008249,0.850433,0.002829,0.920000,98.000000,65.206667


In [25]:
# create a column "token_update" that is "max" if unmask_token is MDM_max or if change_token is change_max, "categorical" if unmask_token is MDM_categorical or if change_token is change_categorical, and "none" if both are none
def determine_token_update(row):
    if row['score_mask_position'] == 'MDM_max' or row['change_token'] == 'change_max':
        return 'max'
    elif row['score_mask_position'] == 'MDM_categorical' or row['change_token'] == 'change_categorical':
        return 'categorical'
    else:
        return 'none'
df['token_update'] = df.apply(determine_token_update, axis=1)
df

,checkpoint,strategy,score_mask_position,select_position,unmask_token,k,gumbel_noise_coefficient,dataset,num_samples,num_denoising_steps,...,self_correction,select_position_unmask,accuracy_mean,accuracy_std,correctly_filled_cells_mean,correctly_filled_cells_std,nfe,time,speed,token_update
0,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,NaN,top_k_gumbel,NaN,1,0,easy,6400,80,...,none,NaN,0.970767,0.002103,0.987500,0.000700,1.083333,252.666667,25.353333,max
1,checkpoints/gidd_0_2/300_epochs,gidd_change_based_on_model_confidence_to_change,NaN,top_k_gumbel,NaN,1,0,hard,6400,80,...,none,NaN,0.679533,0.003350,0.854267,0.001914,1.250000,315.000000,20.316667,max
2,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,MDM_max,NaN,MDM_max,1,0,easy,6400,80,...,NaN,top_k_gumbel,0.967033,0.002335,0.987133,0.000764,1.213333,300.666667,21.296667,max
3,checkpoints/gidd_0_2/300_epochs,gidd_change_low_confidence_positions,MDM_max,NaN,MDM_max,1,0,hard,6400,80,...,NaN,top_k_gumbel,0.665200,0.004419,0.852767,0.001877,1.250000,329.666667,19.510000,max
4,checkpoints/mdlm/300_epochs,mdlm_adaptive_score_select_update,MDM_max,top_k_gumbel,MDM_max,1,0,easy,6400,80,...,NaN,NaN,0.983967,0.001301,0.993767,0.000503,0.920000,97.666667,65.553333,max
5,checkpoints/mdlm/300_epochs,mdlm_adaptive_score_select_update,MDM_max,top_k_gumbel,MDM_max,1,0,hard,6400,80,...,NaN,NaN,0.652300,0.008249,0.850433,0.002829,0.920000,98.000000,65.206667,max


In [26]:
grouped_df = df.groupby(['checkpoint', 'strategy', 'dataset'])['accuracy_mean'].mean().unstack()#.sort_values(by='accuracy_mean')
grouped_df

dataset                                                                              easy  \
checkpoint                      strategy                                                    
checkpoints/gidd_0_2/300_epochs gidd_change_based_on_model_confidence_to_change  0.970767   
                                gidd_change_low_confidence_positions             0.967033   
checkpoints/mdlm/300_epochs     mdlm_adaptive_score_select_update                0.983967   

dataset                                                                              hard  
checkpoint                      strategy                                                   
checkpoints/gidd_0_2/300_epochs gidd_change_based_on_model_confidence_to_change  0.679533  
                                gidd_change_low_confidence_positions             0.665200  
checkpoints/mdlm/300_epochs     mdlm_adaptive_score_select_update                0.652300